In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:


# Count ratings for each user in the original dataset
df_user_count = df_ratings.groupby('user').size().reset_index(name='counts')
df_user_filter = df_user_count[df_user_count['counts'] > 200]

# Count ratings for each ISBN in the original dataset
df_isbn_count = df_ratings.groupby('isbn').size().reset_index(name='counts')
df_isbn_filter = df_isbn_count[df_isbn_count['counts'] > 100]

# Keep ratings from qualified users and books
df_ratings_filtered = df_ratings[
    df_ratings['user'].isin(df_user_filter['user'])
]
df_ratings_filtered = df_ratings_filtered[
    df_ratings_filtered['isbn'].isin(df_isbn_filter['isbn'])
]

# Merge ratings with book titles
df_merged = pd.merge(
    left=df_ratings_filtered,
    right=df_books,
    on='isbn'
)

# Remove duplicate ratings by the same user for the same title
df_merged = df_merged.drop_duplicates(subset=['user', 'title'])

# Create the book-user matrix
df_pivoted = pd.pivot(
    df_merged,
    index='title',
    columns='user',
    values='rating'
)

df_final = df_pivoted.fillna(0)

# Train the KNN model
nn = NearestNeighbors(metric='cosine')
nn.fit(df_final)


In [ ]:


def get_recommends(book=""):
    input_book = df_final.loc[book]

    distances, indices = nn.kneighbors(
        [input_book],
        n_neighbors=6
    )

    recommended_books = [book, []]

    for i in range(5, 0, -1):
        recommended_books[1].append([
            df_final.index[indices[0][i]],
            float(distances[0][i])
        ])

    return recommended_books




In [ ]:

books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
    test_pass = True
    recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")

    if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
        test_pass = False

    recommended_books = ["I'll Be Seeing You", "The Weight of Water", "The Surgeon", "I Know This Much Is True"]
    recommended_books_dist = [0.8, 0.77, 0.77, 0.77]

    for i in range(2):
        if recommends[1][i][0] not in recommended_books:
            test_pass = False

        if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
            test_pass = False

    if test_pass:
        print("You passed the challenge! 🎉🎉🎉🎉🎉")
    else:
        print("You haven't passed yet. Keep trying!")

test_book_recommendation()
